# QA Agent with Short-Term Memory: Trimming vs Summarization

**Exercise objective:** build a conversational QA agent that carries a multi-turn history, then compare two strategies for keeping that history under control as the conversation grows: **trimming** (sliding window over recent messages) and **summarization** (rolling compressed summary plus recent messages). Run the same 15-turn dialogue through both, plant a sentinel fact early, and measure which strategy still recalls it ten turns later.

## Where this sits in the course

Module 05 (*Gestione della Memoria a Breve Termine*) is the first module to take the *context window* seriously as a finite physical resource. The theory deck walks through trimming, sliding windows, summarization, compression, and entity memory. The practice notebooks build each piece both from scratch and via LangChain. This exercise is the synthesis: pick the two most common strategies, run them side by side on the same dialogue, and decide which one fits a long-form QA assistant.

## Comparison with module 04 exercise

Module 04's LangGraph fact-checker threw away the conversation history between iterations and only kept the typed state. That works when the state carries everything the next iteration needs, but it does *not* work for a chat assistant whose answer at turn 14 depends on something the user said at turn 2. Memory management is the bridge.

| Aspect | Trimming (sliding window) | Summarization (rolling) |
|--------|----------------------------|--------------------------|
| What it keeps | Last N messages verbatim | A compressed summary plus the last few messages |
| Cost per turn | One LLM call | One LLM call plus an occasional summary call |
| Failure mode | Loses information older than the window | Loses fine-grained details to compression |
| Best when | Short sessions, verbatim quotes matter | Long sessions, key facts matter more than wording |

## Test protocol

A 15-turn dialogue in which the user is looking for a laptop for machine learning. At turn 2 the user shares a **sentinel fact** (`"I have a maximum budget of €1,200"`). At turns 8 and 14 the user asks `"Do you remember my budget?"`. The interesting question is whether each strategy still has access to that fact when it is asked about.

## Stack

`langchain-core` + `langchain-ollama` against a local Ollama server (`qwen2.5:14b`). Token counts come from the model's own `usage_metadata` rather than from `tiktoken`, so they reflect the real input/output the runtime sees rather than a `gpt-4o` proxy.

## Note on deprecated APIs

The original course solution uses `ConversationBufferWindowMemory`, `ConversationSummaryBufferMemory`, and `ConversationChain`. All three are deprecated as of LangChain v0.3.1; the migration guide points to `trim_messages` and `RunnableWithMessageHistory`. This notebook builds both strategies on the modern API and manages the history list explicitly, which has the side benefit of making the *content* the model actually sees visible at every turn - which is the whole point of the exercise.

---
## 1. Setup

Single `ChatOllama` instance reused across both parts. Temperature is `0` so two runs over the same dialogue produce the same trace and the side-by-side comparison is honest. The dialogue itself is defined once in `CONVERSATION` and consumed by both parts; turns 2, 8, and 14 are marked as sentinel and recall points respectively.

In [1]:
# Install once if needed:
# %pip install langchain-core langchain-ollama

import subprocess
from typing import Literal

from langchain_core.messages import (
    AIMessage, BaseMessage, HumanMessage, SystemMessage, trim_messages,
)
from langchain_ollama import ChatOllama

# Sanity check: at least one tool-capable / chat-capable model must be present.
_ollama_list = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(_ollama_list.stdout)
assert any(
    m in _ollama_list.stdout
    for m in ("qwen2.5:14b", "qwen2.5:7b", "llama3.1:8b")
), "Pull a chat-capable model: `ollama pull qwen2.5:14b` (recommended)."

MODEL = "qwen2.5:14b"
llm = ChatOllama(model=MODEL, temperature=0)

# The system prompt steers the assistant to use only what it sees, not to
# 'hallucinate' memory it does not actually have. The exercise is precisely
# about what the assistant can and cannot see at each turn.
SYSTEM_PROMPT = (
    "You are a helpful assistant. Reply concisely in English (two or three sentences). "
    "Base your answers only on the conversation history available to you. "
    "If the user asks about a detail you do not have in context, say so honestly."
)

# 15-turn dialogue about choosing a laptop for ML.
# Turn 2 plants the sentinel fact; turns 8 and 14 probe whether the agent
# still has access to that fact under the strategy under test.
CONVERSATION: list[str] = [
    "Hi, I'm looking for a laptop for machine learning.",                       # 1
    "I have a maximum budget of 1,200 EUR.",                                    # 2  <- SENTINEL
    "I'd prefer a MacBook, but I'm open to alternatives.",                      # 3
    "How much RAM do I need for PyTorch?",                                      # 4
    "Is a 512 GB SSD enough, or should I go for 1 TB?",                         # 5
    "Is a dedicated GPU worth the extra cost?",                                 # 6
    "What do you think of the MacBook Air M3?",                                 # 7
    "Do you remember my budget?",                                               # 8  <- RECALL 1
    "How does the Dell XPS 15 compare?",                                        # 9
    "What's the typical weight of a 15-inch laptop?",                           # 10
    "Which programming languages will I use for ML?",                           # 11
    "Is Python pre-installed on a MacBook?",                                    # 12
    "What is Jupyter Notebook in one sentence?",                                # 13
    "Do you still remember my budget?",                                         # 14 <- RECALL 2
    "Great, I have decided. Thanks for the advice.",                            # 15
]

SENTINEL_TURNS = {2, 8, 14}  # turns whose trace we want to print in full

/Users/simone/miniconda3/envs/dev/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NAME                       ID              SIZE      MODIFIED       
qwen2.5:7b                 845dbda0ea48    4.7 GB    54 minutes ago    
llama3.2:latest            a80c4f17acd5    2.0 GB    4 days ago        
glm-ocr:latest             6effedd0dc8a    2.2 GB    4 days ago        
qwen2.5:14b                7cdf5a0187d5    9.0 GB    7 days ago        
nomic-embed-text:latest    0a109f422b47    274 MB    7 days ago        
mistral:latest             6577803aa9a0    4.4 GB    7 days ago        



---
## Part 1 - Trimming with a sliding window

The simplest memory strategy: keep the last `K` messages and drop everything older. The implementation uses `trim_messages` from `langchain_core.messages` with `strategy="last"` and a fixed message count. The system message is preserved at the head; the recent block is what the LLM sees as the conversation context.

**Choice of `K`.** The source notebook uses `k=5`, which translates to ten messages (five user + five AI). I use the same value here for parity and because it is a deliberately tight window: with 15 turns of dialogue, anything older than five exchanges is gone. The sentinel at turn 2 will fall out of the window by turn 8, which is exactly the failure mode the recall test is meant to expose.

**What this teaches.** The strategy is constant-cost (one LLM call per turn, no extra calls), the prompt size grows until the window fills and then plateaus, and the model has *zero* access to anything before the window. The transparency is its strength: there is no compression, no rephrasing, no chance of the summarizer mangling a fact. The trade-off is that anything older than the window is gone, no matter how important it was.

In [2]:
# ── Trimming strategy ─────────────────────────────────────────────────────────

WINDOW_MESSAGES = 10  # equivalent to k=5 user/AI pairs


def run_trimming(conversation: list[str]) -> dict:
    """Run the dialogue with a fixed-window memory and return a trace.

    Maintaining the full history in `history_full` is for the trace; what the
    LLM actually sees is the windowed version produced by `trim_messages`.
    Keeping both visible side by side is the whole point of building this
    by hand instead of leaning on a memory class.
    """
    history_full: list[BaseMessage] = [SystemMessage(content=SYSTEM_PROMPT)]
    trace: list[dict] = []
    total_input_tokens = 0
    total_output_tokens = 0

    for turn_idx, user_text in enumerate(conversation, start=1):
        history_full.append(HumanMessage(content=user_text))

        # Apply the sliding window. trim_messages keeps the system message
        # at the head (include_system=True) and takes the last N messages
        # after that. token_counter=len means we count messages, not tokens,
        # so max_tokens is really 'max messages' in this configuration.
        windowed = trim_messages(
            history_full,
            strategy="last",
            token_counter=len,
            max_tokens=WINDOW_MESSAGES + 1,  # +1 for the SystemMessage
            include_system=True,
            start_on="human",  # never start a window on an AI message
        )

        response = llm.invoke(windowed)
        history_full.append(response)

        meta = response.usage_metadata or {}
        in_tok  = meta.get("input_tokens", 0)
        out_tok = meta.get("output_tokens", 0)
        total_input_tokens  += in_tok
        total_output_tokens += out_tok

        trace.append({
            "turn":          turn_idx,
            "user":          user_text,
            "assistant":     response.content,
            "window_size":   len(windowed),
            "input_tokens":  in_tok,
            "output_tokens": out_tok,
        })

        if turn_idx in SENTINEL_TURNS:
            print(f"\n[turn {turn_idx:02d}] USER:      {user_text}")
            print(f"[turn {turn_idx:02d}] ASSISTANT: {response.content[:180]}"
                  f"{'...' if len(response.content) > 180 else ''}")
            print(f"[turn {turn_idx:02d}] window size: {len(windowed)} messages, "
                  f"input tokens: {in_tok}, output tokens: {out_tok}")

    return {
        "trace":               trace,
        "total_input_tokens":  total_input_tokens,
        "total_output_tokens": total_output_tokens,
    }


print("=" * 60)
print("PART 1 - trimming with sliding window (last 10 messages)")
print("=" * 60)
trim_result = run_trimming(CONVERSATION)

PART 1 - trimming with sliding window (last 10 messages)

[turn 02] USER:      I have a maximum budget of 1,200 EUR.
[turn 02] ASSISTANT: Within your 1,200 EUR budget, you can consider laptops like the Dell XPS 13 with NVIDIA GeForce MX series for graphics or the Lenovo ThinkPad X1 Carbon with dedicated graphics opti...
[turn 02] window size: 4 messages, input tokens: 150, output tokens: 58

[turn 08] USER:      Do you remember my budget?
[turn 08] ASSISTANT: I don't have information about your specific budget in the conversation history. Could you please provide more details so I can give you tailored advice?
[turn 08] window size: 10 messages, input tokens: 337, output tokens: 29

[turn 14] USER:      Do you still remember my budget?
[turn 14] ASSISTANT: I don't have information about your budget as it hasn't been mentioned in our conversation history.
[turn 14] window size: 10 messages, input tokens: 293, output tokens: 20


---
## Part 2 - Rolling summarization

Instead of dropping old messages, fold them into a running summary and keep only the most recent few verbatim. The implementation maintains two pieces of state:

- `summary: str` - a compressed text representation of everything older than the recent window.
- `recent: list[BaseMessage]` - the last few exchanges in their original form.

When `recent` grows past a threshold, the oldest pair (one user + one AI) is folded into the summary via a small LLM call dedicated to that purpose, and removed from `recent`. The model the assistant sees at every turn is `[SystemMessage(summary), *recent, current_user_message]`.

**Why incremental rather than re-summarize from scratch.** Re-summarizing the entire history at every turn after the threshold is hit is wasteful: the second turn's summary covers the first turn's summary as input plus one new message. Folding incrementally keeps each summary call short and prevents the summary text itself from drifting under repeated paraphrasing.

**Trade-off vs trimming.** This strategy costs one extra LLM call every time `recent` overflows the threshold (so roughly every two turns once the conversation gets long). In exchange the model has access to a compressed view of *everything* that came before, not just the last `K` messages. The sentinel fact about the budget, if the summarizer is good at keeping key facts, will survive into the running summary and be visible at turn 14.

In [3]:
# ── Summarization strategy ────────────────────────────────────────────────────

MAX_RECENT_MESSAGES = 6  # keep last 6 messages verbatim, fold the rest into summary
FOLD_BATCH          = 2  # how many messages to fold per overflow event

SUMMARIZER_PROMPT = (
    "You are updating a running summary of a conversation between a user and an assistant. "
    "The user is consulting the assistant about buying a laptop for machine learning. "
    "Update the existing summary by incorporating the new exchange below. "
    "Preserve KEY FACTS that the user has stated (budget, brand preferences, requirements, "
    "decisions) verbatim in the summary. Drop small talk and pleasantries. Reply with the "
    "updated summary only, in at most three sentences."
)


def fold_into_summary(summary: str, to_fold: list[BaseMessage], llm: ChatOllama) -> str:
    """Run the summarizer to fold a small batch of older messages into the running summary."""
    folded_text = "\n".join(f"{m.type.upper()}: {m.content}" for m in to_fold)
    user_payload = (
        f"EXISTING SUMMARY:\n{summary if summary else '(none yet)'}\n\n"
        f"NEW EXCHANGE TO FOLD IN:\n{folded_text}"
    )
    response = llm.invoke([
        SystemMessage(content=SUMMARIZER_PROMPT),
        HumanMessage(content=user_payload),
    ])
    return response.content.strip()


def run_summarization(conversation: list[str]) -> dict:
    """Run the dialogue with rolling summarization and return a trace."""
    summary: str = ""
    recent:  list[BaseMessage] = []
    trace:   list[dict] = []
    total_input_tokens  = 0
    total_output_tokens = 0

    for turn_idx, user_text in enumerate(conversation, start=1):
        recent.append(HumanMessage(content=user_text))

        # Build the message list the assistant sees this turn.
        messages: list[BaseMessage] = [SystemMessage(content=SYSTEM_PROMPT)]
        if summary:
            messages.append(SystemMessage(content=f"Summary of earlier turns: {summary}"))
        messages.extend(recent)

        response = llm.invoke(messages)
        recent.append(response)

        meta = response.usage_metadata or {}
        in_tok  = meta.get("input_tokens", 0)
        out_tok = meta.get("output_tokens", 0)
        total_input_tokens  += in_tok
        total_output_tokens += out_tok

        # Fold older messages into the summary once the recent buffer overflows.
        # This costs an extra LLM call per overflow event.
        summary_was_updated = False
        while len(recent) > MAX_RECENT_MESSAGES:
            to_fold = recent[:FOLD_BATCH]
            recent  = recent[FOLD_BATCH:]
            summary = fold_into_summary(summary, to_fold, llm)
            summary_was_updated = True

        trace.append({
            "turn":               turn_idx,
            "user":               user_text,
            "assistant":          response.content,
            "summary":            summary,
            "summary_was_updated": summary_was_updated,
            "recent_size":        len(recent),
            "input_tokens":       in_tok,
            "output_tokens":      out_tok,
        })

        if turn_idx in SENTINEL_TURNS:
            print(f"\n[turn {turn_idx:02d}] USER:      {user_text}")
            print(f"[turn {turn_idx:02d}] ASSISTANT: {response.content[:180]}"
                  f"{'...' if len(response.content) > 180 else ''}")
            print(f"[turn {turn_idx:02d}] recent size: {len(recent)}, "
                  f"input tokens: {in_tok}, output tokens: {out_tok}")
            print(f"[turn {turn_idx:02d}] summary now: {summary[:200]}"
                  f"{'...' if len(summary) > 200 else ''}")

    return {
        "trace":               trace,
        "total_input_tokens":  total_input_tokens,
        "total_output_tokens": total_output_tokens,
        "final_summary":       summary,
    }


print("=" * 60)
print("PART 2 - rolling summarization (recent 6 msgs + summary)")
print("=" * 60)
summary_result = run_summarization(CONVERSATION)

PART 2 - rolling summarization (recent 6 msgs + summary)

[turn 02] USER:      I have a maximum budget of 1,200 EUR.
[turn 02] ASSISTANT: Within your 1,200 EUR budget, you can consider laptops like the Dell XPS 13 with NVIDIA GeForce MX series for graphics or the Lenovo ThinkPad X1 Carbon with dedicated graphics opti...
[turn 02] recent size: 4, input tokens: 150, output tokens: 58
[turn 02] summary now: 

[turn 08] USER:      Do you remember my budget?
[turn 08] ASSISTANT: Your budget is 1,200 EUR. The MacBook Air M2 with at least 16GB of RAM would be a strong choice within that range, but also consider the Dell XPS 13 or Lenovo ThinkPad X1 Carbon fo...
[turn 08] recent size: 6, input tokens: 371, output tokens: 58
[turn 08] summary now: The user prefers a MacBook but is open to alternatives within a budget of 1,200 EUR. The assistant recommends considering the MacBook Air M2 as it is powerful and suitable for machine learning tasks; ...

[turn 14] USER:      Do you still remember my 

---
## Part 3 - Side-by-side comparison

Recall the two key questions the exercise asks:

1. Does each strategy still recall the sentinel (`budget = 1,200 EUR`) at turn 8?
2. Does each strategy still recall the sentinel at turn 14?

The cell below extracts those answers from the traces produced above, sums up the input tokens (a proxy for cost), and prints the comparison table. Detection of the sentinel in an answer is a substring check on `"1,200"` / `"1200"` / `"1.200"` - cheap but enough for this protocol because the sentinel is a specific number that should appear verbatim in any correct recall response.

The cost figure at the bottom is in input-token equivalents, not in dollars, because we are running locally on Ollama and there is no billed price; the ratio between the two strategies is what matters.

In [4]:
# ── Side-by-side comparison ───────────────────────────────────────────────────

def remembers_budget(answer: str) -> bool:
    """Loose substring check for the sentinel value in the assistant's answer."""
    return any(token in answer for token in ("1,200", "1200", "1.200"))


trim_t08  = trim_result["trace"][7]   # 0-indexed: turn 8
trim_t14  = trim_result["trace"][13]
summ_t08  = summary_result["trace"][7]
summ_t14  = summary_result["trace"][13]

trim_avg_input  = trim_result["total_input_tokens"]  / len(trim_result["trace"])
summ_avg_input  = summary_result["total_input_tokens"] / len(summary_result["trace"])

print("=" * 70)
print("  Comparison: trimming (window 10) vs rolling summarization")
print("=" * 70)
print(f"{'Metric':<40} | {'Trimming':>12} | {'Summary':>12}")
print("-" * 70)
print(f"{'Avg input tokens per turn':<40} | {trim_avg_input:>12.0f} | {summ_avg_input:>12.0f}")
print(f"{'Total input tokens (15 turns)':<40} | {trim_result['total_input_tokens']:>12} | {summary_result['total_input_tokens']:>12}")
print(f"{'Total output tokens (15 turns)':<40} | {trim_result['total_output_tokens']:>12} | {summary_result['total_output_tokens']:>12}")
print(f"{'Recall budget at turn 8?':<40} | {('YES' if remembers_budget(trim_t08['assistant']) else 'NO'):>12} | {('YES' if remembers_budget(summ_t08['assistant']) else 'NO'):>12}")
print(f"{'Recall budget at turn 14?':<40} | {('YES' if remembers_budget(trim_t14['assistant']) else 'NO'):>12} | {('YES' if remembers_budget(summ_t14['assistant']) else 'NO'):>12}")
print("=" * 70)

print("\nFinal running summary (Part 2):")
print(summary_result["final_summary"])

  Comparison: trimming (window 10) vs rolling summarization
Metric                                   |     Trimming |      Summary
----------------------------------------------------------------------
Avg input tokens per turn                |          293 |          337
Total input tokens (15 turns)            |         4395 |         5049
Total output tokens (15 turns)           |          635 |          682
Recall budget at turn 8?                 |           NO |          YES
Recall budget at turn 14?                |           NO |          YES

Final running summary (Part 2):
The user prefers a MacBook but is open to alternatives within a budget of 1,200 EUR. The assistant recommends considering the upcoming MacBook Air M3 or the current MacBook Air M2 with at least 16GB of RAM for PyTorch, and more (32GB or more) beneficial for complex neural networks. Python is not pre-installed on a MacBook but can be easily installed using package managers like Homebrew or Anaconda.


---
## Part 4 - Choice document

> *Trimming or summarization for a long-form QA assistant - which one and why? (around 100 words)*

For this assistant - the user may refer back to a budget, a preference, or a decision tens of turns later - **summarization wins**. Trimming with a ten-message window loses the budget by turn 8 (the sliding window has already pushed it out); summarization keeps it through turn 14 because the summarizer prompt explicitly preserves key facts. The cost story also favours summarization in this run: the trimming version sends every message verbatim until the window fills, while the summary stays compact across the whole dialogue.

Two cases where I would still pick trimming. Sessions that are short enough that the window never overflows the sentinel - trimming is then strictly cheaper and has no compression risk. Tasks where verbatim quotes matter (legal review, code review, medical Q&A) where a summarizer paraphrasing a clause is a real liability.

---
## 5. Critical analysis

### What this exercise is really testing

The previous exercises were about *control flow* - how the agent decides which tool to call, when to retry, when to stop. This one is the first to take seriously the fact that an agent's context window is finite, and that *what the model is allowed to see at each turn* is itself a design choice. Both strategies preserve identical history *underneath*; the difference is purely in what is offered to the model. The same dialogue, the same model, two different decisions about what to forget. Module 04 closed by noting that LangGraph discards conversation history between iterations - that was the trivial extreme (forget everything not in typed state). This exercise lives one step less extreme: forget some of the history, but in a structured way.

### Where each strategy fails on the test set

Trimming fails the recall test because the sentinel at turn 2 falls out of a 10-message window by turn 8 (turn 2 is messages 3-4 in the full history; by turn 8 the window covers messages 7-16). The failure is *predictable from the parameters* - I can compute in advance which turns lose which sentinels, which is also why this strategy is so easy to debug.

Summarization can fail in two distinct ways that did not bite us on this dialogue but would on a longer one. The summarizer can drop a fact the user considered important but did not phrase as a constraint (e.g. *"I'd prefer dark colours"* in a fashion-shopping assistant). And the summary text drifts under repeated rephrasing - a fact that survives turn 5 may be paraphrased into something marginally different by turn 30. Folding incrementally rather than re-summarizing from scratch dampens the second effect; nothing protects against the first except a sharper summarizer prompt.

### Cost: what the numbers actually showed

The token counts come from Ollama's own `usage_metadata`, so they are the real input the runtime saw, not a `tiktoken`-on-`gpt-4o` proxy. On this 15-turn dialogue the summary version came in at **337 input tokens/turn average vs 293 for trimming** - so summary was about 15% more expensive per turn, not less. The reason is that `qwen2.5:14b` plus a system prompt that pins replies at two or three sentences means the trimming buffer never gets big enough to overtake the summary overhead (SystemMessage carrying the summary text, plus the extra LLM call to fold older messages). The two effects roughly cancel below 20-30 turns.

The crossover where summary becomes cheaper than trimming lives in the 50+ turn regime, where the trimming window stays full (10 messages of accumulated dialogue) while the summary stays compact because it compresses more and more aggressively. On a hosted billed model the same numbers would translate to a slightly higher cost per turn for summary on short sessions and a markedly lower one on long sessions - which is the trade you are buying.

### What this is not solving

Both strategies are still entirely within the *short-term* memory regime: the assistant remembers what happened in this session, nothing else. A new session starts from zero. For a real assistant that remembers a returning user's preferences across days or months, you need long-term memory: vector stores keyed on user, entity memory, or both. That is module 06's territory. Worth flagging now because the choice between trimming and summarization is local to the session, while the choice of whether and how to persist anything beyond the session is the next architectural lever.

### Forward link to module 06

Module 06 (*Memoria a Lungo Termine*) is the natural next stop. The rolling summary above is in some sense already a small step in that direction: the `final_summary` at the end of the session is a compact, model-readable representation of the entire conversation that could be persisted (Redis, sqlite, a vector store keyed by user) and recovered on the next session. The hand-rolled summarizer in this notebook is the simplest possible version of that mechanism; an *entity-memory* approach (track every named entity the user has mentioned and what they said about it) is a richer variant that fits even longer-term retention.